<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_76_classical_rl_foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🎮 **Module 76 — Classical RL Foundations (MDP → PPO)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 10 · Master-Plan Gaps


# Module 76 — Classical RL Foundations

> M67's RLHF / GRPO assumed you knew RL. This module fills that in **from first principles**. We'll walk: **MDP → Bellman → value iteration → tabular Q-learning → DQN → REINFORCE → Actor-Critic → PPO**. At the end, you'll see why **PPO** is the algorithm RLHF picked, why **DPO** (M62) reframed the problem as supervised learning, and why **GRPO** (M67) drops the value function entirely.

### What you'll cover
1. The RL setting — agent, environment, reward
2. **MDPs** — `(S, A, P, R, γ)` and the discounted return
3. **Bellman equations** — the recursive structure that makes RL tractable
4. **Value iteration** in a 4×4 GridWorld (no neural net)
5. **Tabular Q-learning** — TD(0), ε-greedy, off-policy
6. **DQN** — Q-learning with neural networks (Atari-era)
7. **Policy gradients** — REINFORCE on CartPole
8. **Actor-Critic / A2C** — variance reduction with a baseline
9. **PPO** — the clipped surrogate that powers ChatGPT-era RLHF
10. **Connecting back to M62 / M67** — DPO and GRPO as RL specialisations


## 1 · The RL setting

```
                    ┌─────────────────────┐
                    │      AGENT          │
                    │  (policy π_θ)       │
                    └────────┬────────────┘
                action a_t   │   ▲   reward r_t , next state s_{t+1}
                             ▼   │
                    ┌─────────────────────┐
                    │   ENVIRONMENT       │
                    └─────────────────────┘
```

At each timestep:
1. The **environment** is in some state `s_t`.
2. The **agent** picks an action `a_t ∼ π_θ(·|s_t)` (its **policy**).
3. The environment responds with `(r_t, s_{t+1})`.
4. Repeat until episode end.

The agent's **goal** is to maximise expected cumulative reward. Everything in RL is a different recipe for **how to find the policy that does that.**

## 2 · Markov Decision Processes

A finite **MDP** is a 5-tuple `(S, A, P, R, γ)`:

| Symbol | Meaning |
|---|---|
| `S` | set of states |
| `A` | set of actions |
| `P(s' | s, a)` | transition probability |
| `R(s, a)` | reward function |
| `γ ∈ [0, 1)` | discount factor — how much we care about future reward |

**Markov property**: the future depends only on `s_t` and `a_t`, not on the history before. This is what makes RL tractable — we don't have to track the full trajectory.

**Return** from time `t`:
$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \ldots = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$$

`γ` close to 0 = myopic (only the next reward matters). `γ` close to 1 = far-sighted. Standard for most problems: `γ = 0.99`.

## 3 · Bellman equations — the recursion

Define two **value functions**:
- **State value** `V^π(s)` = expected return starting from `s` following policy `π`.
- **Action value** `Q^π(s, a)` = expected return after taking `a` in `s`, then following `π`.

They satisfy **Bellman's equations**:

$$V^{\pi}(s) = \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a) \big[R(s,a) + \gamma V^{\pi}(s')\big]$$

$$Q^{\pi}(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \sum_{a'} \pi(a'|s') Q^{\pi}(s',a')$$

And the **Bellman optimality** equation for the *best* policy:

$$Q^{*}(s, a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \max_{a'} Q^{*}(s', a')$$

**Why this matters.** Every classical RL algorithm is a different recipe for **iteratively solving** one of these equations.

## 4 · Value iteration in GridWorld

Tiny 4×4 grid. Start top-left, goal bottom-right. Each step costs −1; reaching the goal ends the episode. Four actions: up, down, left, right. Walls bounce you back.

In [ ]:
!pip -q install numpy matplotlib torch gymnasium

In [ ]:
import numpy as np
np.set_printoptions(precision=2, suppress=True)

ROWS, COLS = 4, 4
GOAL = (3, 3)
ACTIONS = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}
GAMMA = 0.99

def step(state, action):
    r, c = state
    dr, dc = ACTIONS[action]
    nr, nc = min(max(r + dr, 0), ROWS - 1), min(max(c + dc, 0), COLS - 1)
    nstate = (nr, nc)
    reward = 0.0 if nstate == GOAL else -1.0
    done = nstate == GOAL
    return nstate, reward, done

# Initialise V(s) = 0
V = np.zeros((ROWS, COLS))
for sweep in range(50):
    new_V = V.copy()
    for r in range(ROWS):
        for c in range(COLS):
            if (r, c) == GOAL: continue
            best = -np.inf
            for a in ACTIONS:
                (nr, nc), reward, done = step((r, c), a)
                best = max(best, reward + GAMMA * V[nr, nc] * (0.0 if done else 1.0))
            new_V[r, c] = best
    if np.allclose(new_V, V, atol=1e-6): print(f"converged at sweep {sweep}"); V = new_V; break
    V = new_V
print(V)

**Read the output.** The number in each cell is the value `V*(s)` — the expected discounted return from that cell under the optimal policy. Closer to the goal → less negative → better.

Two algorithms hide in this loop:
- **Value iteration**: update `V ← max_a [R + γ · V(s')]` each sweep until convergence.
- **Policy iteration**: alternate (1) **evaluate** `V^π` for fixed `π`, (2) **improve** `π ← argmax_a Q^π(s, a)`. Same fixed point.

In a real environment we don't know `P` and `R` up front. We have to **learn them by interaction**. That's where Q-learning starts.

## 5 · Tabular Q-learning

**TD(0)** update — adjust `Q(s,a)` toward the *bootstrapped* one-step return:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \big[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \big]$$

This is the famous **off-policy** TD update. The trick that makes it work in practice: **ε-greedy** exploration — take the best action `1 - ε` of the time, a random action `ε` of the time. Without exploration, the agent gets stuck.

In [ ]:
import random

Q = np.zeros((ROWS, COLS, len(ACTIONS)))
ACT_LIST = list(ACTIONS.keys())
EPS = 0.2
LR  = 0.1

def ep_greedy(s, eps):
    if random.random() < eps:
        return random.randint(0, len(ACT_LIST) - 1)
    return int(np.argmax(Q[s[0], s[1]]))

for episode in range(2000):
    s = (0, 0)
    for _ in range(50):
        ai = ep_greedy(s, EPS)
        s2, r, done = step(s, ACT_LIST[ai])
        target = r + GAMMA * (0.0 if done else Q[s2[0], s2[1]].max())
        Q[s[0], s[1], ai] += LR * (target - Q[s[0], s[1], ai])
        s = s2
        if done: break

print("learned policy (greedy direction per cell):")
print(np.vectorize(lambda i: ACT_LIST[i])(np.argmax(Q, axis=-1)))

You should see arrows pointing roughly toward the goal — the agent **learned the optimal policy by interaction**, with no access to `P` or `R` up front. Two key properties of Q-learning:

- **Off-policy**: the data-collection policy (ε-greedy) can be different from the policy we're learning (greedy on `Q`).
- **Bootstrapping**: the target uses our own `Q` estimate (`max_a Q(s', a')`) — risky in theory, robust in practice.

## 6 · From table to network — DQN

Tabular Q-learning blows up when `|S|` is huge (Atari pixels, robot joints, language). **Function approximation** with a neural net solves it:

$$Q_\theta(s, a) \approx Q^*(s, a)$$

The training objective is the same TD target, now a regression loss:

$$\mathcal{L}(\theta) = \mathbb{E}\big[(r + \gamma \max_{a'} Q_{\theta^-}(s', a') - Q_\theta(s, a))^2\big]$$

Two stabilisation tricks DeepMind found in 2013-15:
1. **Replay buffer** — train on randomly-sampled past transitions, not just the latest one. Decorrelates gradients.
2. **Target network** `Q_{θ^-}` — a slow-moving copy used for the TD target. Without this, the bootstrap chases its own tail and diverges.

Modern descendants — **Double DQN**, **Dueling DQN**, **Rainbow DQN**, **C51** distributional RL — are tweaks on this skeleton.

## 7 · Policy gradients — REINFORCE

Q-learning learns a value function and *derives* a policy. **Policy gradient** methods directly learn a parameterised policy `π_θ(a|s)` (e.g. a small neural net that outputs action logits).

The **policy gradient theorem** says:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\!\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

In plain English: **for each step in a trajectory, multiply `∇log π(a|s)` by the return `G_t` and add it up.** That's the gradient that pushes your policy toward higher reward.

In [ ]:
import gymnasium as gym
import torch, torch.nn as nn, torch.nn.functional as F

env = gym.make("CartPole-v1")
torch.manual_seed(0)

class Policy(nn.Module):
    def __init__(self, in_dim=4, out_dim=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.Tanh(),
                                 nn.Linear(hidden, out_dim))
    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)

pi = Policy()
opt = torch.optim.Adam(pi.parameters(), lr=3e-3)

def run_episode():
    obs, _ = env.reset(seed=None)
    log_probs, rewards = [], []
    for _ in range(500):
        x = torch.tensor(obs, dtype=torch.float32)
        probs = pi(x)
        m = torch.distributions.Categorical(probs)
        a = m.sample()
        log_probs.append(m.log_prob(a))
        obs, r, term, trunc, _ = env.step(int(a.item()))
        rewards.append(r)
        if term or trunc: break
    return log_probs, rewards

GAMMA_ = 0.99
for ep in range(200):
    log_probs, rewards = run_episode()
    # discounted returns G_t
    Gs = []
    g = 0
    for r in reversed(rewards):
        g = r + GAMMA_ * g
        Gs.insert(0, g)
    Gs = torch.tensor(Gs, dtype=torch.float32)
    Gs = (Gs - Gs.mean()) / (Gs.std() + 1e-8)   # variance reduction
    loss = -sum(lp * G for lp, G in zip(log_probs, Gs))
    opt.zero_grad(); loss.backward(); opt.step()
    if ep % 20 == 0: print(f"ep {ep:>3}  reward={sum(rewards):.0f}  loss={loss.item():.2f}")

env.close()

**Read the trace.** Reward should climb from ~20 (random) to ~200-500 over a few hundred episodes. That's CartPole solved by **REINFORCE**.

Notice three things:
1. We **standardise** the returns (`G - mean(G) / std(G)`). Pure REINFORCE has notoriously high variance — this is a cheap baseline.
2. The loss has a **`-`** in front. PyTorch minimises; we want to *maximise* expected return.
3. We need a **full episode** before doing one update — no bootstrap, no TD. Slow.

That last property is what motivates Actor-Critic.

## 8 · Actor-Critic — variance reduction with a critic

REINFORCE's variance bites at scale. The fix: subtract a **baseline** `b(s)` from the return. The gradient is **unbiased** as long as the baseline doesn't depend on the action.

The natural baseline is `V(s)` — the value function. The remainder `G_t - V(s_t)` is called the **advantage** `A_t`:

$$\nabla_\theta J = \mathbb{E}\!\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot \hat{A}_t\right]$$

Concretely, we train **two networks**:
- **Actor** `π_θ(a|s)` — the policy.
- **Critic** `V_φ(s)` — the baseline, trained with MSE on observed returns.

This is **A2C** (synchronous actor-critic). A few enhancements get you to the SOTA-shape of modern RL:

| Trick | Name |
|---|---|
| Run many envs in parallel | **A2C** (sync), **A3C** (async) |
| Use a multi-step return (n-step TD) | reduces bias-variance trade-off |
| Use **Generalised Advantage Estimation (GAE)** | exponentially-weighted mix of 1..N-step TD; tunable via `λ` |
| Add an entropy bonus to the loss | encourages exploration |
| Clip the policy update (next section) | **PPO** |

GAE is the single most important trick. Its formula:
$$\hat{A}_t^{GAE(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}, \quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

`λ=0` recovers TD(0); `λ=1` recovers Monte-Carlo. `λ=0.95` is the production default.

## 9 · PPO — proximal policy optimisation

Vanilla policy gradient has a stability problem: one large gradient step can ruin the policy permanently. **PPO** (Schulman et al., 2017) fixes that with a **clipped surrogate loss**:

$$\mathcal{L}_{PPO}(\theta) = \mathbb{E}_t\!\left[ \min\!\big( \rho_t \hat{A}_t,\; \text{clip}(\rho_t, 1-\epsilon, 1+\epsilon)\hat{A}_t \big) \right]$$

where the **importance ratio** is

$$\rho_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$$

**The clip is the magic.** If the new policy's probability for the chosen action moves more than a factor `1±ε` from the old policy's, the gradient is clipped — you can't over-update on a single batch.

Full PPO loss:
$$\mathcal{L} = \mathcal{L}_{PPO} - c_1 \mathcal{L}_V + c_2 \mathcal{H}[\pi_\theta]$$

where `L_V` is the value-function MSE (critic) and `H[π]` is an entropy bonus. Typical: `ε = 0.1-0.2`, `c_1 = 0.5`, `c_2 = 0.01`, `γ = 0.99`, `λ = 0.95`.

In [ ]:
# Minimal PPO update skeleton — for full code see stable-baselines3 / CleanRL.
ppo_sketch = '''
# Rollout phase: collect (s, a, r, log_prob_old, value_old) for T steps
# Then K epochs of mini-batch updates:

for epoch in range(K_EPOCHS):
    for mb in mini_batches(rollout):
        # 1. Compute current policy log-probs + values
        new_logp = policy.log_prob(mb.actions, mb.states)
        new_value = critic(mb.states)

        # 2. Importance ratio
        ratio = (new_logp - mb.old_logp).exp()

        # 3. Advantage (GAE), normalised
        adv = mb.advantages

        # 4. Clipped surrogate
        unclipped = ratio * adv
        clipped   = ratio.clamp(1 - EPS, 1 + EPS) * adv
        policy_loss = -torch.min(unclipped, clipped).mean()

        # 5. Value loss
        value_loss = F.mse_loss(new_value, mb.returns)

        # 6. Entropy bonus (encourages exploration)
        entropy = policy.entropy(mb.states).mean()

        # 7. Total
        loss = policy_loss + 0.5 * value_loss - 0.01 * entropy
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
        opt.step()
'''
print(ppo_sketch)

**That's modern RL in 30 lines.** Production libraries — **stable-baselines3**, **CleanRL**, **RLlib**, **TorchRL** — wrap exactly this loop with logging, checkpointing, env vectorisation, and a few extra tricks (advantage clipping, target value clipping, learning-rate annealing).

## 10 · Connecting back to M62 (DPO) and M67 (GRPO)

Now you can read M62 and M67 *as RL algorithms*, not as magic LLM tricks.

### RLHF / PPO (the original)
1. **Collect human preferences** `(prompt, chosen, rejected)`.
2. Train a **reward model** `r_θ(x, y)` that predicts which side wins (M67 §3).
3. Treat **text generation** as RL: state = prompt so far, action = next token, reward = `r_θ(full response) − β · KL(π‖π_ref)`.
4. Apply **PPO** (this module §9) to the LLM. The "actor" is the policy LLM; the "critic" is a separately-trained value head.

The two LLM-specific quirks vs vanilla PPO:
- Per-token reward is the KL penalty; sequence-level reward is the RM score.
- Reward is **sparse** (only at the end of the sequence) and noisy.

### DPO (M62) — RL without RL
Bradley-Terry preferences + the closed-form KL-constrained optimal policy collapse the entire PPO loop into **one supervised loss**:
$$\mathcal{L}_{DPO} = -\log \sigma\big(\beta \big[\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\big]\big)$$

No reward model, no value model, no rollouts, no clipping. *Same effective gradient* as PPO with a perfect reward model. That's why DPO blew up post-2023.

### GRPO (M67) — same PPO loss, group-relative baseline
Drop the value model. For each prompt, sample `G` completions, score each with a scalar reward function. The **advantage** of each is the *standardised* reward inside the group:
$$\hat{A}_i = \frac{r_i - \text{mean}(r_{1..G})}{\text{std}(r_{1..G})}$$

Plug that into PPO's clipped surrogate loss. **GRPO ≡ PPO with the group acting as the critic.** That's why DeepSeek-R1 reached frontier reasoning quality without a separate value network — the group baseline is enough.

## ✅ Recap
- RL is **agent + environment + reward**. Find the policy that maximises expected discounted return.
- **MDPs** make it tractable (Markov property). **Bellman equations** give us the recursion.
- **Value iteration / policy iteration** solve known-MDP problems.
- **Q-learning** + ε-greedy learns from interaction; **DQN** scales it to neural nets.
- **REINFORCE** is the simplest policy-gradient method — high variance, needs full episodes.
- **Actor-Critic** + **GAE** + parallel envs is the recipe behind every modern RL paper.
- **PPO** is Actor-Critic + a *clipped surrogate* that prevents catastrophic updates. It's the algorithm RLHF picked.
- **DPO** (M62) collapses RLHF into a supervised loss. **GRPO** (M67) keeps PPO's clipping but uses a group baseline instead of a value model.

Next: **M77 — GPU Networking (NVLink · InfiniBand · RoCE)**.
